# Random Forest - Base P-COLD

Mémoire M2 EGRF - Détection et pilotage du risque opérationnel bancaire

## Objectif

On reprend la même variable cible que pour le modèle logit (fraude vs non-fraude) et les mêmes variables explicatives (`Business line`, `log_loss`), afin de comparer directement les deux approches. Le Random Forest sert ici de **modèle intermédiaire** : plus flexible que le logit, mais encore partiellement interprétable via l'importance des variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, accuracy_score, ConfusionMatrixDisplay)

plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
df = pd.read_csv('pcold_clean.csv')
df['Y_fraude'] = df['Event_type_grouped'].isin(['Internal fraud', 'External fraud']).astype(int)
print(f"{len(df)} événements chargés.")

## 1. Préparation des données

On garde exactement les mêmes variables explicatives que pour le logit (`Business line` encodée en dummies, `log_loss`), pour que la comparaison entre les deux modèles soit directe et honnête.

In [ ]:
variables_explicatives = ['Business line', 'log_loss']
df_modele = df[variables_explicatives + ['Y_fraude']].dropna()

X = pd.get_dummies(df_modele[['Business line']], drop_first=True).astype(float)
X['log_loss'] = df_modele['log_loss']
y = df_modele['Y_fraude']

print(f"N utilisées : {len(df_modele)}")
print(f"Répartition de Y_fraude : {y.value_counts(normalize=True).round(3).to_dict()}")
# On garde ce chiffre en tête : environ 1/3 de non-fraude, 2/3 de fraude.
# Le déséquilibre est présent mais pas extrême sur cette cible binaire.

## 2. Découpage train / test

Contrairement au logit (estimé sur l'ensemble de l'échantillon, dans une logique inférentielle classique), le Random Forest est évalué selon une logique prédictive : on entraîne le modèle sur une partie des données (`train`) et on vérifie ses performances sur des données qu'il n'a jamais vues (`test`).

`stratify=y` garantit que la proportion de fraude/non-fraude est respectée à l'identique dans les deux sous-échantillons.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train : {len(X_train)} observations")
print(f"Test  : {len(X_test)} observations")

## 3. Entraînement du Random Forest

Quelques choix de paramètres à noter :
- `n_estimators=500` : nombre d'arbres agrégés (B dans la méthodologie)
- `class_weight='balanced'` : pondère automatiquement les classes en fonction de leur fréquence inverse, pour compenser le déséquilibre entre fraude et non-fraude (voir méthodologie, robustesse au déséquilibre)
- `min_samples_leaf=5` : évite que les arbres ne deviennent trop spécifiques (surapprentissage) en imposant un minimum de 5 observations par feuille
- `random_state=42` : graine aléatoire fixée pour la reproductibilité des résultats

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1  # utilise tous les coeurs disponibles pour accélérer l'entraînement
)
rf.fit(X_train, y_train)

# Prédictions sur l'échantillon test
y_pred = rf.predict(X_test)                 # classe prédite (0 ou 1)
y_proba = rf.predict_proba(X_test)[:, 1]     # probabilité prédite d'être une fraude

print("Modèle entraîné.")

## 4. Évaluation des performances

Le rapport de classification donne, pour chaque classe, la précision (parmi les événements prédits comme fraude, combien le sont réellement), le rappel (parmi les fraudes réelles, combien sont détectées), et le F1-score (moyenne harmonique des deux). C'est plus informatif que la seule accuracy en présence de déséquilibre entre classes.

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Non-fraude', 'Fraude']))

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
auc = roc_auc_score(y_test, y_proba)
print(f"AUC      : {auc:.4f}")

In [ ]:
# Matrice de confusion, même style visuel que le modèle logit pour comparaison directe
matrice = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=matrice, display_labels=['Non-fraude', 'Fraude'])
disp.plot(cmap='Blues', values_format='d')
plt.title("Matrice de confusion - Random Forest")
plt.show()

In [ ]:
# Courbe ROC, même style que le logit
taux_fp, taux_vp, seuils = roc_curve(y_test, y_proba)

plt.plot(taux_fp, taux_vp, color='#1f77b4', linewidth=2, label=f'Courbe ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='#a0a0a0', linestyle='--', label='Modèle aléatoire')
plt.fill_between(taux_fp, taux_vp, alpha=0.1, color='#1f77b4')
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs")
plt.title("Courbe ROC - Random Forest")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Importance des variables

Contrairement au logit qui donne un coefficient et un sens d'effet pour chaque variable, le Random Forest ne donne qu'une mesure d'**importance relative** : à quel point chaque variable contribue, en moyenne sur tous les arbres, à réduire l'impureté de Gini (voir méthodologie). Elle ne dit rien sur le sens de l'effet (positif ou négatif), seulement sur son poids global dans les prédictions.

In [ ]:
importance = pd.DataFrame({
    'Variable': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

importance

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(importance['Variable'], importance['Importance'], color='#1f77b4')
plt.xlabel("Importance (réduction moyenne de l'impureté de Gini)")
plt.title("Importance des variables - Random Forest")
plt.gca().invert_yaxis()  # variable la plus importante en haut
plt.tight_layout()
plt.show()

**Lecture** : `log_loss` domine largement l'importance des variables, suivie par les lignes métier `Retail banking` et `Asset management`. Ce classement est cohérent avec les résultats du modèle logit, où `log_loss` était également la variable la plus significative et `Retail banking` affichait l'odds ratio le plus élevé.